# Steel Defect Detection — YOLO11 Fine-Tuning (Colab)

Fine-tune **YOLO11n** on the [NEU Surface Defect Database](https://www.kaggle.com/datasets/ucirvine/steel-defect-detection) (NEU-DET) using transfer learning from a COCO-pretrained checkpoint.

**Workflow:** download dataset → convert VOC XML to YOLO format → train → evaluate → upload weights to Hugging Face.

> Run this notebook on Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU).

## 1. Environment setup

In [ ]:
!pip install -q ultralytics huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone project & add dataset

Download NEU-DET from Kaggle and place it at `data/NEU-DET/` (same layout as the local repo), or copy it from Drive.

In [ ]:
!git clone https://github.com/aimanahmed495-max/steel-defect-detection.git
%cd steel-defect-detection

In [ ]:
# Example: copy dataset from Google Drive (adjust path to your upload)
# !cp -r /content/drive/MyDrive/NEU-DET data/NEU-DET

## 3. Convert PASCAL VOC annotations to YOLO format

The conversion script maps NEU-DET's six defect classes and writes `data/yolo/data.yaml` for Ultralytics.

In [ ]:
!python -m src.convert_annotations

## 4. Fine-tune YOLO11

Transfer learning from `yolo11n.pt` (COCO-pretrained). Hyperparameters are tuned for a small industrial dataset — adjust epochs and image size as needed.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="data/yolo/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project="runs/detect",
    name="train",
    exist_ok=True,
)

## 5. Validate on the held-out split

In [ ]:
best = YOLO("runs/detect/train/weights/best.pt")
metrics = best.val(data="data/yolo/data.yaml")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## 6. Visualize predictions

In [ ]:
from pathlib import Path
from IPython.display import Image, display

sample = next(Path("data/yolo/images/val").glob("*.jpg"))
pred = best.predict(str(sample), save=True, conf=0.25)
display(Image(filename=pred[0].save_dir + "/" + sample.name))

## 7. Upload weights to Hugging Face

Create a model repo on Hugging Face, then upload `best.pt`. Update `models/README.md` in the project with your repo ID.

In [ ]:
from huggingface_hub import HfApi

HF_REPO = "YOUR_USERNAME/steel-defect-yolo11"  # change before running

api = HfApi()
api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
api.upload_file(
    path_or_fileobj="runs/detect/train/weights/best.pt",
    path_in_repo="best.pt",
    repo_id=HF_REPO,
    repo_type="model",
)
print(f"Uploaded to https://huggingface.co/{HF_REPO}")

## 8. Save to Drive (optional backup)

In [ ]:
# !cp runs/detect/train/weights/best.pt /content/drive/MyDrive/steel-defect-best.pt